# Stage 2 — BindCraft cloud smoke test (EGFR challenge)

Purpose: distinguish **ENVIRONMENT FAILURE** from **EGFR TARGET FAILURE**.

1. Smallest official example: PDL1, hotspot 56, 65 aa, cap 1 relaxed trajectory.
2. Micro EGFR run: Domain III crop 310-481, approved envelope B, conservative hotspot `390,393,399,421,424,431`, 80 aa, cap 3 relaxed trajectories.

Rules baked into this notebook: no target mutation, no hard-target settings, official default 4stage-multimer advanced/filter files unchanged (only `max_trajectories`), no paid compute.

**Before running:** Runtime -> Change runtime type -> T4 GPU (free tier is fine for the smoke; if you are offered L4/A100, record it).

Run cells top to bottom. Do NOT change any numbers. If a cell OOMs twice, stop and report — do not keep crashing the runtime.

Pinned: BindCraft `7713aa0d0d351e4117a8befeb8541f3a8ebd3368`. Weights: official AlphaFold `alphafold_params_2022-12-06.tar` (~5.3 GB).

## Cell 1 — baseline hardware metadata

In [ ]:
import json, os, platform, shutil, subprocess, sys, time

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

def gpu_query(field):
    return sh(f"nvidia-smi --query-gpu={field} --format=csv,noheader,nounits 2>/dev/null")

env_pre = {
    'colab_python': sys.version.split()[0],
    'platform': platform.platform(),
    'gpu_name': gpu_query('name') or 'NO NVIDIA GPU VISIBLE',
    'gpu_vram_total_mib': int(x) if (x := gpu_query('memory.total')) else None,
    'gpu_vram_free_mib_at_start': int(x) if (x := gpu_query('memory.free')) else None,
    'host_ram_total_gb': round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1),
    'disk_free_gb_content': round(shutil.disk_usage('/content').free / 1e9, 1),
    'nvcc': sh('nvcc --version | tail -1') or None,
    'nvidia_driver': sh('nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null') or None,
}
print(json.dumps(env_pre, indent=2))
assert env_pre['gpu_name'] != 'NO NVIDIA GPU VISIBLE', 'GPU runtime not enabled! Runtime -> Change runtime type -> GPU, then reconnect.'

## Cell 2 — install official BindCraft (pinned commit) + ColabDesign + PyRosetta + AF2 weights
Mirrors the official Colab install cell; only difference is the pinned commit. Takes ~5-15 min; weights download runs in parallel.

In [ ]:
PINNED = '7713aa0d0d351e4117a8befeb8541f3a8ebd3368'
if not os.path.isfile('/content/bindcraft/.pinned_sha'):
    print('cloning BindCraft pinned commit')
    assert sh('git clone https://github.com/martinpacesa/BindCraft /content/bindcraft') == '' or os.path.exists('/content/bindcraft/bindcraft.py')
    sh(f'cd /content/bindcraft && git checkout {PINNED}')
    os.chmod('/content/bindcraft/functions/dssp', 0o755)
    os.chmod('/content/bindcraft/functions/DAlphaBall.gcc', 0o755)
    print('installing ColabDesign (BindCraft install script uses unpinned main; resolved version is recorded below)')
    print(sh('pip install -q git+https://github.com/sokrypton/ColabDesign.git'))
    print('installing PyRosetta (academic/non-commercial use per its license)')
    print(sh('pip install -q pyrosetta --find-links https://west.rosettacommons.org/pyrosetta/quarterly/release.cxx11thread.serialization'))
    print('downloading AlphaFold params in background')
    sh('apt-get install aria2 -qq')
    sh('mkdir -p /content/bindcraft/params && cd /content/bindcraft && (aria2c -q -x 16 -d params https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar && tar -xf params/alphafold_params_2022-12-06.tar -C params && touch params/done.txt)&')
    open('/content/bindcraft/.pinned_sha','w').write(PINNED)

sha = sh('cd /content/bindcraft && git rev-parse HEAD')
assert sha == PINNED, f'unexpected checkout {sha}'
print('BindCraft at', sha)
print('waiting for params...')
for _ in range(360):
    if os.path.isfile('/content/bindcraft/params/done.txt'):
        break
    time.sleep(5)
assert os.path.isfile('/content/bindcraft/params/done.txt'), 'params download did not finish'
print(sorted(os.listdir('/content/bindcraft/params')))

## Cell 3 — record post-install environment (JAX/CUDA/GPU, resolved versions)

In [ ]:
def pkg_version(name):
    out = sh(f'pip show {name} 2>/dev/null | grep -i "^Version"')
    return out.split(':',1)[1].strip() if out else None

import jax
env_post = {
    **env_pre,
    'bindcraft_commit': sh('cd /content/bindcraft && git rev-parse HEAD'),
    'colabdesign_pip_version': pkg_version('colabdesign'),
    'colabdesign_location': (lambda out: out.split(':',1)[1].strip() if out else None)(sh('pip show colabdesign 2>/dev/null | grep -i Location')),
    'jax_version': jax.__version__,
    'jaxlib_version': pkg_version('jaxlib'),
    'jax_devices': [f'{d.platform}:{d.device_kind}' for d in jax.devices()],
    'cuda_nvcc': env_pre['nvcc'],
    'biopython': pkg_version('biopython'),
    'pyrosetta_installed': pkg_version('pyrosetta') is not None,
    'af_weights_source': 'https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar',
    'af_weights_files': sorted(f for f in os.listdir('/content/bindcraft/params') if f.endswith('.npz')),
    'gpu_vram_free_mib_postinstall': int(x) if (x := gpu_query('memory.free')) else None,
    'disk_free_gb_postinstall': round(shutil.disk_usage('/content').free / 1e9, 1),
}
os.makedirs('/content/drive/MyDrive/BindCraft/stage2_smoke', exist_ok=True) if os.path.exists('/content/drive') else None
print(json.dumps(env_post, indent=2))

## Cell 4 — mount Google Drive (results persist beyond the Colab session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNROOT = '/content/drive/MyDrive/BindCraft/stage2_smoke'
os.makedirs(RUNROOT, exist_ok=True)
json.dump(env_post, open(f'{RUNROOT}/env_metadata.json','w'), indent=2)
print('env metadata saved to', f'{RUNROOT}/env_metadata.json')

## Cell 5 — upload target crop PDB and the analyzer script
When the file chooser opens, select both files from your local clone:
- `data/processed/6ARU_chainA_domain3_310-481.pdb`
- `scripts/analyze_bindcraft_run.py`

In [ ]:
from google.colab import files
os.makedirs('/content/egfr', exist_ok=True)
up = files.upload()
for name, data in up.items():
    dest = f'/content/egfr/{name}' if name.endswith('.pdb') else f'/content/{name}'
    open(dest,'wb').write(data)
    print('saved', dest)

# verify crop: 172 residues, UniProt numbering 310-481, hotspot identities
res = {}
for line in open('/content/egfr/6ARU_chainA_domain3_310-481.pdb'):
    if line.startswith('ATOM') and line[12:16].strip() == 'CA':
        res[int(line[22:26])] = line[17:20].strip()
aa3 = {'Q':'GLN','E':'GLU','D':'ASP','K':'LYS','R':'ARG','N':'ASN','T':'THR'}
expect = {390:'Q',391:'E',393:'D',396:'K',399:'K',421:'E',422:'N',424:'E',427:'R',429:'R',430:'T',431:'K'}
assert sorted(res) == list(range(310,482)), f'crop numbering wrong: {len(res)} residues'
assert all(res[p] == aa3[a] for p,a in expect.items()), 'hotspot identity mismatch'
print('crop verified:', len(res), 'residues; all 12 exposed-residue identities match')

## Cell 6 — write smoke configs into the cloned BindCraft tree
Target JSONs are byte-equivalent in content to the repo `configs/bindcraft/*.json`. Advanced settings are **derived from the official default file at the pinned commit** with only `max_trajectories` changed; the cell asserts nothing else differs.

In [ ]:
targets = {
 'pdl1_official_smoke.json': {
    'design_path': f'{RUNROOT}/pdl1_official_smoke/', 'binder_name': 'PDL1_smoke',
    'starting_pdb': '/content/bindcraft/example/PDL1.pdb', 'chains': 'A',
    'target_hotspot_residues': '56', 'lengths': [65,65], 'number_of_final_designs': 1},
 'egfr_d3_B_conservative.json': {
    'design_path': f'{RUNROOT}/egfr_d3_B_conservative/', 'binder_name': 'EGFR_D3_Bcons',
    'starting_pdb': '/content/egfr/6ARU_chainA_domain3_310-481.pdb', 'chains': 'A',
    'target_hotspot_residues': '390,393,399,421,424,431', 'lengths': [80,80],
    'number_of_final_designs': 1},
}
os.makedirs('/content/bindcraft/settings_target', exist_ok=True)
for name, d in targets.items():
    json.dump(d, open(f'/content/bindcraft/settings_target/{name}','w'), indent=4)

def smoke_advanced(max_traj):
    base = json.load(open('/content/bindcraft/settings_advanced/default_4stage_multimer.json'))
    assert base['max_trajectories'] is False and base['predict_initial_guess'] is False
    base['max_trajectories'] = max_traj
    return base

json.dump(smoke_advanced(1), open('/content/bindcraft/advanced_smoke_max1.json','w'), indent=4)
json.dump(smoke_advanced(3), open('/content/bindcraft/advanced_smoke_max3.json','w'), indent=4)
# prove only max_trajectories differs from the official default
default = json.load(open('/content/bindcraft/settings_advanced/default_4stage_multimer.json'))
for f, n in [('advanced_smoke_max1.json',1), ('advanced_smoke_max3.json',3)]:
    d2 = json.load(open(f'/content/bindcraft/{f}'))
    diff = {k:(default[k],d2[k]) for k in default if default[k]!=d2[k]}
    assert diff == {'max_trajectories': (False, n)}, diff
print('configs written; only max_trajectories differs from official defaults')

## Cell 7 — run helper: timed subprocess + 2 s VRAM polling
Then execute the **smallest official example** (PDL1, cap 1 relaxed trajectory). If this fails, the problem is the environment, not EGFR.

In [ ]:
import threading
def _poll_vram(path, stop):
    with open(path,'w') as f:
        while not stop.is_set():
            used = gpu_query('memory.used')
            f.write(f'{time.time():.0f},{used}\n'); f.flush(); time.sleep(2)

def run_job(tag, settings, advanced):
    log, vram = f'/content/{tag}.log', f'/content/{tag}_vram.csv'
    stop = threading.Event(); th = threading.Thread(target=_poll_vram, args=(vram, stop)); th.start()
    t0 = time.time()
    with open(log,'w') as lf:
        p = subprocess.run(['python','-u','bindcraft.py',
                            '--settings', settings,
                            '--filters','./settings_filters/default_filters.json',
                            '--advanced', advanced],
                           cwd='/content/bindcraft', stdout=lf, stderr=subprocess.STDOUT, text=True)
    wall = time.time()-t0; stop.set(); th.join()
    vals = [int(l.split(',')[1]) for l in open(vram) if ',' in l and l.split(',')[1].strip().isdigit()]
    rec = {'tag':tag,'returncode':p.returncode,'wall_s':round(wall,1),
           'peak_vram_mib':max(vals) if vals else None,'log':log,'vram_log':vram}
    print(json.dumps(rec, indent=2))
    return rec

job_records = {}
job_records['pdl1_official_smoke'] = run_job(
    'pdl1_official_smoke',
    '/content/bindcraft/settings_target/pdl1_official_smoke.json',
    '/content/bindcraft/advanced_smoke_max1.json')
print('---- last 30 log lines ----')
print('\n'.join(open('/content/pdl1_official_smoke.log').read().splitlines()[-30:]))

## Cell 8 — STOP AND CHECK
Only continue to the EGFR cell if the PDL1 job actually produced a relaxed trajectory (returncode 0 and a PDB under `.../pdl1_official_smoke/Trajectory/Relaxed/`). If it failed:
- OOM: note free/total VRAM and the failing step, retry **once** after Runtime->Factory reset; a second OOM ends the smoke (do not relaunch repeatedly).
- any other error: save the log, do not touch EGFR configs, report environment failure.

In [ ]:
pdl1_dir = f'{RUNROOT}/pdl1_official_smoke'
relaxed = os.path.exists(f'{pdl1_dir}/Trajectory/Relaxed') and \
          [f for f in os.listdir(f'{pdl1_dir}/Trajectory/Relaxed') if f.endswith('.pdb')]
print('PDL1 relaxed trajectories:', relaxed)
assert relaxed, 'official example did not produce a relaxed trajectory -> environment failure, stop here'

## Cell 9 — micro EGFR run: envelope B, conservative 6-residue hotspot, 80 aa, cap 3
This is not a production campaign. `number_of_final_designs=1` and `max_trajectories=3` bound the run. A completed trajectory is not a successful binder.

In [ ]:
job_records['egfr_d3_B_conservative'] = run_job(
    'egfr_d3_B_conservative',
    '/content/bindcraft/settings_target/egfr_d3_B_conservative.json',
    '/content/bindcraft/advanced_smoke_max3.json')
print('---- last 40 log lines ----')
print('\n'.join(open('/content/egfr_d3_B_conservative.log').read().splitlines()[-40:]))

## Cell 10 — geometry analysis vs approved envelope B + report bundle

In [ ]:
import importlib.util, sys
spec = importlib.util.spec_from_file_location('analyze_bindcraft_run', '/content/analyze_bindcraft_run.py')
analyzer = importlib.util.module_from_spec(spec); sys.modules['analyze_bindcraft_run'] = analyzer; spec.loader.exec_module(analyzer)

analyses = {}
for tag in ['pdl1_official_smoke','egfr_d3_B_conservative']:
    rdir = f'{RUNROOT}/{tag}'
    out = f'{RUNROOT}/{tag}_geometry.json'
    sys.argv = ['analyze_bindcraft_run.py','--run-dir',rdir,'--out',out]
    try:
        analyzer.main()
        analyses[tag] = json.load(open(out))['summary']
    except Exception as e:
        analyses[tag] = {'analysis_error': repr(e)}

report = {'env': env_post, 'jobs': job_records, 'geometry': analyses,
          'notes': 'completed trajectory != successful binder; official filters decide acceptance'}
json.dump(report, open(f'{RUNROOT}/stage2_smoke_report.json','w'), indent=2)
print(json.dumps(analyses, indent=2))
print('REPORT:', f'{RUNROOT}/stage2_smoke_report.json')

## Cell 11 — visual inspection of every completed EGFR trajectory
Check: binder near the intended surface, no severe clash, no binder hugging the crop edge, physically reasonable orientation. Cartoon only — use the geometry JSON for numbers.

In [ ]:
import glob, py3Dmol
pdbs = sorted(glob.glob(f'{RUNROOT}/egfr_d3_B_conservative/Trajectory/Relaxed/*.pdb')) + \
       sorted(glob.glob(f'{RUNROOT}/egfr_d3_B_conservative/Trajectory/LowConfidence/*.pdb')) + \
       sorted(glob.glob(f'{RUNROOT}/egfr_d3_B_conservative/Trajectory/Clashing/*.pdb'))
print('showing', len(pdbs), 'trajectories')
for pdb in pdbs:
    v = py3Dmol.view(width=500, height=350)
    v.addModel(open(pdb).read(), 'pdb')
    v.setStyle({'chain':'A'}, {'cartoon': {'color':'#3c5b6f'}})
    v.setStyle({'chain':'B'}, {'cartoon': {'color':'#B76E79'}})
    # envelope B sidechains as yellow sticks
    for r in list(range(390,404))+list(range(421,432)):
        v.addStyle({'chain':'A','resi':r}, {'stick':{'color':'yellow'}})
    v.zoomTo({'chain':'B'}); v.setTitle(os.path.basename(pdb))
    v.show()

## Done — what to send back
Share the Drive folder `BindCraft/stage2_smoke/` (or at minimum `env_metadata.json`, `stage2_smoke_report.json`, both `*.log` files and one relaxed PDB). Keep the runtime running if anything needs a quick re-check; otherwise Factory reset frees the GPU.

Do not run the broad hotspot config or increase trajectory counts until the smoke report is reviewed. No paid compute without written approval.